# Week 8 — Final evaluation and the research report

## Overview

This week completes the eight-week research study. We reproduce the agreed development
procedure, lock every setting, generate all final predictions, and only then evaluate
the held-out contracts. The outcome is a reproducible comparison, not a promised victory
for machine learning.

**Learning objectives**

1. Separate prediction error from Monte Carlo reference noise using an analytic audit.
2. Distinguish held-out contract error, test-subportfolio error and full-portfolio error.
3. Interpret accuracy, simulation work, total runtime and learned-neighbor quality together.
4. Write evidence-based findings, limitations and presentation-ready figures.

**Prerequisites:** the Week 7 design freeze and the earlier pricing/learning material.
**Workload:** one 75-minute meeting and approximately 4–6 hours outside the meeting.

**Outputs:** final result tables, held-out predictions, three research figures, an
independent MC audit, learned weights, a freeze manifest, and a short report outline.
All required code is embedded. No previous notebook files are imported.

## 1. What the final experiment can establish

We price a terminal GMAB gross top-up with a specified fee under a lognormal risk-neutral
model, assuming survival to maturity. This intentionally simple product has an analytic
put-option formula. **Monte Carlo is used here to study a workflow intended for more
complex liabilities; this project does not claim a practical speed advantage over the
analytic formula for this product.**

The analytic values are an audit instrument. Models receive only simulated reference
and validation labels. This lets us study label noise and approximation error separately.
The analytic formula is the natural computational benchmark for this particular toy
product; report its evaluation time alongside the Monte Carlo/learning comparison.

Learning a distance may improve neighbor selection yet fail to improve total price
accuracy when label noise or sparse coverage dominates. A direct value predictor may
work well without offering an interpretable notion of contract similarity. These are
different empirical findings.

In [ ]:
# Complete common pricing code. No data files or earlier notebooks are required.
import sys
import json
import platform
from pathlib import Path
from time import perf_counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import norm

try:
    from IPython.display import display
except ImportError:
    def display(*objects):
        for obj in objects:
            print(obj)

RISK_FREE_RATE = 0.03
FEATURES = ['account_ratio', 'guarantee_ratio', 'maturity', 'volatility', 'fee']
plt.rcParams.update({'figure.figsize': (7.2, 4.3), 'axes.grid': True,
                     'grid.alpha': 0.22, 'font.size': 10})


def make_portfolio(n=1200, seed=2027):
    """Synthetic in-force GMAB contracts, never customer data.

    P is a reference premium, A0=P*account_ratio, and G=P*guarantee_ratio.
    T is the remaining term. All contracts share the same risk-free rate.
    Volatility describes the contract's fund; rates are annual decimals.
    """
    if not isinstance(n, (int, np.integer)) or n < 10:
        raise ValueError('n must be an integer of at least 10.')
    rng = np.random.default_rng(seed)
    return pd.DataFrame({
        'contract_id': np.arange(n, dtype=int),
        'premium': rng.uniform(10_000.0, 50_000.0, n),
        'account_ratio': rng.uniform(0.65, 1.40, n),
        'guarantee_ratio': rng.uniform(0.90, 1.60, n),
        'maturity': rng.integers(5, 21, n).astype(float),
        'volatility': rng.uniform(0.12, 0.35, n),
        'fee': rng.uniform(0.005, 0.030, n),
    })


def validate_contracts(df):
    required = ['contract_id', 'premium'] + FEATURES
    missing = [c for c in required if c not in df]
    if missing:
        raise ValueError(f'Missing contract columns: {missing}')
    if len(df) == 0:
        raise ValueError('Supply at least one contract.')
    values = df[required].to_numpy(dtype=float)
    if not np.isfinite(values).all():
        raise ValueError('All contract fields must be finite.')
    ids = df['contract_id'].to_numpy(dtype=float)
    if (ids < 0).any() or (ids != np.floor(ids)).any() or len(np.unique(ids)) != len(ids):
        raise ValueError('contract_id must contain unique nonnegative integers.')
    if (df[['premium', 'account_ratio', 'guarantee_ratio']] <= 0).any().any():
        raise ValueError('Premium and account/guarantee ratios must be positive.')
    if (df[['maturity', 'volatility', 'fee']] < 0).any().any():
        raise ValueError('Maturity, volatility and fee must be nonnegative.')


def gmab_mc(df, n_paths=2048, seed=123):
    """Independent plain Monte Carlo for per-premium gross top-up liability.

    Identical (seed, contract_id) reproduces the same path stream even when
    the contract appears in a different subset or row order. Distinct IDs
    give independent streams. This is NOT a common-market portfolio-risk
    simulator: it estimates a sum of expected liabilities, not portfolio VaR.
    SE describes Monte Carlo sampling uncertainty only.
    """
    validate_contracts(df)
    if not isinstance(n_paths, (int, np.integer)) or n_paths < 2:
        raise ValueError('n_paths must be an integer of at least 2.')
    if not isinstance(seed, (int, np.integer)) or seed < 0:
        raise ValueError('seed must be a nonnegative integer.')
    values, errors, sds = [], [], []
    for row in df.itertuples():
        rng = np.random.default_rng(np.random.SeedSequence([int(seed), int(row.contract_id)]))
        z = rng.standard_normal(n_paths)
        terminal_ratio = row.account_ratio * np.exp(
            (RISK_FREE_RATE - row.fee - 0.5 * row.volatility**2) * row.maturity
            + row.volatility * np.sqrt(row.maturity) * z
        )
        payoff = np.exp(-RISK_FREE_RATE * row.maturity) * np.maximum(
            row.guarantee_ratio - terminal_ratio, 0.0)
        sd = float(np.std(payoff, ddof=1))
        values.append(float(np.mean(payoff)))
        errors.append(sd / np.sqrt(n_paths))
        sds.append(sd)
    return pd.DataFrame({'value': values, 'se': errors, 'payoff_sd': sds,
                         'paths': int(n_paths)}, index=df.index)


def gmab_exact(df):
    """Closed-form put benchmark under the simplified model: audit only.

    Never use these values as training or tuning labels in the project.
    The deterministic cases T=0 or sigma=0 are handled separately.
    """
    validate_contracts(df)
    a = df['account_ratio'].to_numpy(dtype=float)
    g = df['guarantee_ratio'].to_numpy(dtype=float)
    t = df['maturity'].to_numpy(dtype=float)
    sig = df['volatility'].to_numpy(dtype=float)
    fee = df['fee'].to_numpy(dtype=float)
    ans = np.maximum(g * np.exp(-RISK_FREE_RATE*t) - a * np.exp(-fee*t), 0.0)
    regular = (t > 0) & (sig > 0)
    if np.any(regular):
        tr, sr = t[regular], sig[regular]
        d1 = (np.log(a[regular] / g[regular]) +
              (RISK_FREE_RATE-fee[regular]+0.5*sr**2)*tr) / (sr*np.sqrt(tr))
        d2 = d1 - sr*np.sqrt(tr)
        ans[regular] = (g[regular]*np.exp(-RISK_FREE_RATE*tr)*norm.cdf(-d2)
                        - a[regular]*np.exp(-fee[regular]*tr)*norm.cdf(-d1))
    return np.maximum(ans, 0.0)


def split_portfolio(df, seed=2027):
    """Split contracts before building pairs, selecting labels or tuning."""
    validate_contracts(df)
    rng = np.random.default_rng(seed)
    order = rng.permutation(len(df))
    n_candidate, n_validation = int(0.60*len(df)), int(0.20*len(df))
    if min(n_candidate, n_validation, len(df)-n_candidate-n_validation) < 1:
        raise ValueError('Portfolio is too small for the requested split.')
    return {
        'candidate': df.iloc[order[:n_candidate]].copy(),
        'validation': df.iloc[order[n_candidate:n_candidate+n_validation]].copy(),
        'test': df.iloc[order[n_candidate+n_validation:]].copy(),
    }


def score_predictions(y_true, y_pred, premium=None):
    """Errors per premium, plus signed aggregate relative error when requested."""
    truth = np.asarray(y_true, dtype=float).reshape(-1)
    pred = np.asarray(y_pred, dtype=float).reshape(-1)
    if truth.shape != pred.shape or len(truth) == 0:
        raise ValueError('Truth and predictions must have equal nonzero length.')
    if not (np.isfinite(truth).all() and np.isfinite(pred).all()):
        raise ValueError('Truth and predictions must be finite.')
    error = pred-truth
    out = {'rmse': float(np.sqrt(np.mean(error**2))),
           'mae': float(np.mean(np.abs(error))),
           'p95_abs': float(np.quantile(np.abs(error), 0.95)),
           'bias': float(np.mean(error)),
           'negative_fraction': float(np.mean(pred < 0))}
    if premium is not None:
        p = np.asarray(premium, dtype=float).reshape(-1)
        if p.shape != truth.shape or not np.isfinite(p).all() or np.any(p <= 0):
            raise ValueError('Premium weights must be finite, positive and aligned.')
        denominator = float(p @ truth)
        out['portfolio_relative_error'] = (float(p @ error)/denominator
                                            if abs(denominator) > 1e-12 else np.nan)
    return out


In [ ]:
# Complete shared learning code. All labels are supplied explicitly by the caller.
from scipy.spatial.distance import cdist
from scipy.optimize import minimize
from sklearn.preprocessing import MinMaxScaler, SplineTransformer
from sklearn.cluster import KMeans
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.pipeline import make_pipeline


def build_features(candidate, query=None, columns=FEATURES):
    """Fit feature scales only on the unlabelled candidate pool.

    A constant training column maps to zero on training data. A held-out
    observation outside the training range may map outside [0,1]; no clipping.
    No premium, contract ID or computed valuation is an informative feature.
    """
    scaler = MinMaxScaler()
    z_candidate = scaler.fit_transform(candidate[list(columns)])
    z_query = None if query is None else scaler.transform(query[list(columns)])
    return scaler, z_candidate, z_query


def select_representatives(Z, m, method='kmeans', seed=2027):
    """Return m DISTINCT row positions of actual candidate contracts.

    K-means is fitted to candidate FEATURES only. For each nonempty cluster,
    choose the member closest to that cluster's center. This is a member
    representative, not an exact k-medoids optimization or Feng et al.'s SRSC.
    """
    z = np.asarray(Z, dtype=float)
    if z.ndim != 2 or not np.isfinite(z).all():
        raise ValueError('Z must be a finite 2D matrix.')
    if not isinstance(m, (int, np.integer)) or not 1 <= m <= len(z):
        raise ValueError('m must lie between 1 and the number of candidates.')
    rng = np.random.default_rng(seed)
    if method == 'random':
        return np.sort(rng.choice(len(z), size=m, replace=False))
    if method != 'kmeans':
        raise ValueError("method must be 'random' or 'kmeans'.")
    km = KMeans(n_clusters=m, n_init=5, max_iter=150, random_state=seed)
    membership = km.fit_predict(z)
    chosen = []
    for cluster in range(m):
        members = np.flatnonzero(membership == cluster)
        if len(members):
            distances = np.sum((z[members] - km.cluster_centers_[cluster])**2, axis=1)
            chosen.append(int(members[np.argmin(distances)]))
    # Duplicate feature vectors can yield fewer than m nonempty clusters.
    # Keep the requested contract count without ever repeating an ID.
    if len(chosen) < m:
        remaining = np.setdiff1d(np.arange(len(z)), chosen)
        chosen.extend(rng.choice(remaining, m-len(chosen), replace=False).tolist())
    return np.sort(np.asarray(chosen, dtype=int))


def nearest_predict(Zref, yref, Zquery, weights=None, k=1):
    """Uniform k-neighbor transfer; k=1 is the CSM-style teaching baseline.

    A diagonal weighted distance is ordinary Euclidean distance after each
    coordinate is multiplied by sqrt(weight). Stable sort breaks exact ties
    in reference-row order. Chunking avoids an N-by-N distance matrix.
    """
    zr, zq = np.asarray(Zref, dtype=float), np.asarray(Zquery, dtype=float)
    yr = np.asarray(yref, dtype=float).reshape(-1)
    if zr.ndim != 2 or zq.ndim != 2 or zr.shape[1] != zq.shape[1] or len(zr) != len(yr):
        raise ValueError('Reference/query dimensions or labels do not align.')
    if not (np.isfinite(zr).all() and np.isfinite(zq).all() and np.isfinite(yr).all()):
        raise ValueError('Features and labels must be finite.')
    if not isinstance(k, (int, np.integer)) or not 1 <= k <= len(zr):
        raise ValueError('k must be between 1 and the reference count.')
    w = np.ones(zr.shape[1]) if weights is None else np.asarray(weights, dtype=float)
    if w.shape != (zr.shape[1],) or not np.isfinite(w).all() or np.any(w < 0):
        raise ValueError('Distance weights must be finite and nonnegative.')
    scale = np.sqrt(w)
    neighbors = np.empty((len(zq), k), dtype=int)
    for start in range(0, len(zq), 1024):
        stop = min(start+1024, len(zq))
        distance = cdist(zq[start:stop]*scale, zr*scale, metric='sqeuclidean')
        neighbors[start:stop] = np.argsort(distance, axis=1, kind='stable')[:, :k]
    return yr[neighbors].mean(axis=1), neighbors


def fit_metric(Zref, yref, lam=0.01, max_pairs=2000, seed=2027):
    """Fit a simple diagonal response-aware pseudometric on reference pairs.

    Objective: mean(((H@w)/p - ((y_i-y_j)/s)^2)^2)
               + lam/2 * sum((w-1)^2),  w>=0, sum(w)=p.
    Here H=(z_i-z_j)^2 coordinatewise and s=max(ptp(yref),1e-6).
    The label scale is part of the modeling choice, fitted on references only.
    Numerical labels have noise: this loss is not an unbiased exact-gap loss.
    """
    z, y = np.asarray(Zref, dtype=float), np.asarray(yref, dtype=float).reshape(-1)
    if z.ndim != 2 or len(z) != len(y) or len(y) < 2:
        raise ValueError('Supply at least two aligned labeled reference contracts.')
    if not (np.isfinite(z).all() and np.isfinite(y).all()):
        raise ValueError('Training data must be finite.')
    if lam < 0 or not isinstance(max_pairs, (int, np.integer)) or max_pairs < 1:
        raise ValueError('lam must be nonnegative and max_pairs a positive integer.')
    n, p = z.shape
    count = min(max_pairs, n*(n-1)//2)
    rng = np.random.default_rng(seed)
    if count == n*(n-1)//2:
        i, j = np.triu_indices(n, k=1)
    else:
        # Uniform directed non-self pairs sampled WITH replacement.
        # They share contracts and are not independent observational units.
        i = rng.integers(0, n, size=count)
        j = (i+rng.integers(1, n, size=count)) % n
    label_scale = max(float(np.ptp(y)), 1e-6)
    h = (z[i]-z[j])**2 / p
    target = ((y[i]-y[j])/label_scale)**2
    def objective(w):
        residual = h @ w-target
        return float(np.mean(residual**2) + 0.5*lam*np.sum((w-1.0)**2))
    def gradient(w):
        return 2.0*h.T @ (h @ w-target)/len(target) + lam*(w-1.0)
    result = minimize(objective, np.ones(p), jac=gradient, method='SLSQP',
                      bounds=[(0.0, float(p))]*p,
                      constraints=[{'type': 'eq', 'fun': lambda w: w.sum()-p,
                                    'jac': lambda w: np.ones(p)}],
                      options={'maxiter': 400, 'ftol': 1e-10})
    feasible = (np.isfinite(result.x).all() and result.x.min() >= -1e-7
                and abs(result.x.sum()-p) <= 1e-6)
    success = bool(result.success and feasible)
    weights = np.clip(result.x, 0.0, None) if success else np.ones(p)
    weights *= p/weights.sum()
    return {'weights': weights, 'success': success, 'pair_count': len(target),
            'objective': objective(weights), 'scale': label_scale,
            'iterations': int(result.nit), 'message': str(result.message)}


def make_regressors(seed=2027):
    """Fixed small model menu. Regression predicts values, not distances."""
    return {
        'Ridge': Ridge(alpha=1.0),
        'SplineGAM': make_pipeline(
            SplineTransformer(n_knots=4, degree=3, include_bias=False),
            Ridge(alpha=1.0)),
        'RandomForest': RandomForestRegressor(n_estimators=100, min_samples_leaf=3,
                                               random_state=seed, n_jobs=1),
        'GradientBoosting': GradientBoostingRegressor(n_estimators=100, max_depth=2,
                                                       learning_rate=0.05, random_state=seed),
    }


In [ ]:
# Reproducible benchmark helpers; pricing and learning definitions appear above.
import hashlib
import sklearn
import scipy

DEFAULT_PROTOCOL = {
    'n_contracts': 1200, 'portfolio_seed': 101, 'split_seed': 2027,
    'representative_counts': [40, 80], 'compressors': ['random', 'kmeans'],
    'seeds': [2027, 2028], 'budget': 262144,
    'n_validation_labels': 40, 'validation_paths': 1024,
    'families': ['CSM1NN', 'KNN', 'Ridge', 'SplineGAM', 'RandomForest',
                 'GradientBoosting', 'Learned1NN'],
    'extension': 'none', 'test_audit_paths': 8192,
    'label_target': 'normalized_gross_GMAB_top_up_at_fixed_fee',
    'risk_free_rate': 0.03,
}


def budget_plan(n_total, m, budget, n_val, val_paths):
    inputs = [n_total, m, budget, n_val, val_paths]
    if any(not isinstance(x, (int, np.integer)) or x <= 0 for x in inputs):
        raise ValueError('Budget inputs must be positive integers.')
    b_val = n_val*val_paths
    ref_paths = (budget-b_val)//m
    direct_paths = budget//n_total
    if min(ref_paths, direct_paths, val_paths) < 2:
        raise ValueError('Infeasible budget: require >=2 paths for every MC estimate.')
    return {'reference_paths': int(ref_paths), 'direct_paths': int(direct_paths),
            'reference_work': int(m*ref_paths), 'validation_work': int(b_val),
            'pipeline_work': int(m*ref_paths+b_val),
            'direct_work': int(n_total*direct_paths),
            'pipeline_unused': int(budget-m*ref_paths-b_val),
            'direct_unused': int(budget-n_total*direct_paths)}


def _regression_candidates(family, seed):
    if family == 'Ridge':
        return [(f'alpha={a:g}', Ridge(alpha=a)) for a in [0.1, 1.0, 10.0]]
    if family == 'SplineGAM':
        return [(f'knots=4;alpha={a:g}', make_pipeline(
            SplineTransformer(n_knots=4, degree=3, include_bias=False), Ridge(alpha=a)))
                for a in [0.1, 1.0, 10.0]]
    if family == 'RandomForest':
        return [(f'100trees;leaf={leaf}', RandomForestRegressor(
            n_estimators=100, min_samples_leaf=leaf, random_state=seed, n_jobs=1))
                for leaf in [2, 5]]
    if family == 'GradientBoosting':
        return [('100trees;depth=2;lr=.05', make_regressors(seed)[family])]
    raise ValueError(f'Unknown regression family: {family}')


def develop_one(df, protocol, m, compressor, seed):
    """Fit/tune only on candidate labels and validation labels. No test truth."""
    t0 = perf_counter()
    parts = split_portfolio(df, seed=int(protocol['split_seed']))
    candidate = parts['candidate']
    n_val = int(protocol['n_validation_labels'])
    if n_val > len(parts['validation']):
        raise ValueError('Not enough validation contracts in the fixed split.')
    validation = parts['validation'].iloc[:n_val].copy()
    scaler, z_candidate, z_val = build_features(candidate, validation)
    prep_seconds = perf_counter()-t0
    work = budget_plan(len(df), int(m), int(protocol['budget']), n_val,
                       int(protocol['validation_paths']))
    t0 = perf_counter()
    positions = select_representatives(z_candidate, m, method=compressor, seed=seed)
    representatives = candidate.iloc[positions]
    zr = z_candidate[positions]
    compression_seconds = perf_counter()-t0
    t0 = perf_counter()
    reference_labels = gmab_mc(representatives, work['reference_paths'], seed=seed+10000)
    reference_seconds = perf_counter()-t0
    t0 = perf_counter()
    validation_labels = gmab_mc(validation, int(protocol['validation_paths']), seed=seed+20000)
    validation_seconds = perf_counter()-t0
    yr = reference_labels['value'].to_numpy()
    yv = validation_labels['value'].to_numpy()
    common_seconds = prep_seconds+compression_seconds+reference_seconds+validation_seconds
    models, rows, tuning_rows = {}, [], []
    for family in protocol['families']:
        started = perf_counter()
        candidates = []
        if family in ('CSM1NN', 'KNN'):
            ks = [1] if family == 'CSM1NN' else [k for k in [3, 5, 9] if k <= m]
            for k in ks:
                predict = lambda z, k=k: nearest_predict(zr, yr, z, k=k)[0]
                candidate_record = {'name': f'k={k}', 'predict': predict,
                                    'weights': None, 'k': k, 'metric_success': True}
                candidate_record['validation_mse'] = float(np.mean((predict(z_val)-yv)**2))
                candidates.append(candidate_record)
        elif family == 'Learned1NN':
            for lam in [0.0, 0.0001, 0.01, 0.1]:
                fitted = fit_metric(zr, yr, lam=lam, max_pairs=2000, seed=seed+30000)
                weights = fitted['weights']
                predict = lambda z, w=weights: nearest_predict(zr, yr, z, weights=w)[0]
                candidates.append({'name': f'lambda={lam:g}', 'predict': predict,
                                   'weights': weights, 'k': 1,
                                   'metric_success': fitted['success'],
                                   'metric_diagnostics': fitted,
                                   'validation_mse': float(np.mean((predict(z_val)-yv)**2))})
        else:
            for setting, estimator in _regression_candidates(family, seed):
                estimator.fit(zr, yr)
                prediction = estimator.predict(z_val)
                candidates.append({'name': setting,
                                   'predict': lambda z, model=estimator: model.predict(z),
                                   'weights': None, 'k': None, 'metric_success': True,
                                   'validation_mse': float(np.mean((prediction-yv)**2))})
        best = min(candidates, key=lambda c: c['validation_mse'])
        tuning_seconds = perf_counter()-started
        best['tuning_seconds'] = tuning_seconds
        models[family] = best
        for c in candidates:
            tuning_rows.append({'compressor': compressor, 'm': m, 'seed': seed,
                                'family': family, 'setting': c['name'],
                                'validation_rmse': np.sqrt(c['validation_mse']),
                                'selected': c is best, 'metric_success': c['metric_success']})
        rows.append({'compressor': compressor, 'm': m, 'seed': seed, 'family': family,
                     'setting': best['name'], 'validation_rmse': np.sqrt(best['validation_mse']),
                     'reference_paths': work['reference_paths'],
                     'reference_work': work['reference_work'],
                     'validation_work': work['validation_work'],
                     'pipeline_work': work['pipeline_work'],
                     'unused_work': work['pipeline_unused'],
                     'prep_s': prep_seconds, 'compress_s': compression_seconds,
                     'reference_sim_s': reference_seconds, 'validation_sim_s': validation_seconds,
                     'fit_and_select_s': tuning_seconds,
                     'development_s': common_seconds+tuning_seconds,
                     'metric_success': best['metric_success']})
    return {'rows': rows, 'tuning_rows': tuning_rows, 'models': models,
            'parts': parts, 'scaler': scaler, 'Zref': zr, 'yref': yr,
            'representatives': representatives, 'reference_labels': reference_labels,
            'validation': validation, 'validation_labels': validation_labels,
            'work': work, 'common_seconds': common_seconds,
            'm': m, 'compressor': compressor, 'seed': seed}


def run_development(df, protocol):
    """Shared experiment runtime and reconstructed standalone pipeline costs differ."""
    if len(df) != protocol['n_contracts']:
        raise ValueError('Protocol contract count does not match the supplied portfolio.')
    if float(protocol['risk_free_rate']) != RISK_FREE_RATE:
        raise ValueError('Update the pricing model and protocol together before changing r.')
    if protocol['label_target'] != 'normalized_gross_GMAB_top_up_at_fixed_fee':
        raise ValueError('The configured label target differs from the implemented model.')
    started = perf_counter()
    states = []
    for seed in protocol['seeds']:
        for m in protocol['representative_counts']:
            for compressor in protocol['compressors']:
                states.append(develop_one(df, protocol, int(m), compressor, int(seed)))
    table = pd.DataFrame([row for state in states for row in state['rows']])
    tuning = pd.DataFrame([row for state in states for row in state['tuning_rows']])
    return states, table, tuning, perf_counter()-started


def freeze_manifest(protocol, development_table):
    """Record all candidate rules and validation-selected settings before audit."""
    simple_settings = development_table[['compressor', 'm', 'seed', 'family', 'setting']]
    manifest = {'protocol': json.loads(json.dumps(protocol)),
                'selected_settings': simple_settings.to_dict(orient='records'),
                'hyperparameter_grids': {
                    'KNN_k': [3, 5, 9], 'Ridge_alpha': [0.1, 1.0, 10.0],
                    'SplineGAM_knots': 4, 'SplineGAM_alpha': [0.1, 1.0, 10.0],
                    'RF_leaf': [2, 5], 'RF_trees': 100,
                    'GB_trees': 100, 'GB_depth': 2, 'GB_learning_rate': 0.05,
                    'metric_lambda': [0.0, 0.0001, 0.01, 0.1], 'metric_max_pairs': 2000},
                'test_used_for_selection': False,
                'features': list(FEATURES),
                'metrics': ['test_rmse', 'test_mae', 'test_p95_abs', 'test_bias',
                            'test_subportfolio_relative_error', 'full_portfolio_relative_error',
                            'end_to_end_s', 'simulation_work'],
                'software': {'python': platform.python_version(), 'numpy': np.__version__,
                             'pandas': pd.__version__, 'scipy': scipy.__version__,
                             'scikit_learn': sklearn.__version__}}
    canonical = json.dumps(manifest, sort_keys=True, separators=(',', ':')).encode()
    manifest['sha256'] = hashlib.sha256(canonical).hexdigest()
    return manifest


def verify_manifest(manifest, protocol, development_table):
    payload = {k: v for k, v in manifest.items() if k != 'sha256'}
    canonical = json.dumps(payload, sort_keys=True, separators=(',', ':')).encode()
    if hashlib.sha256(canonical).hexdigest() != manifest.get('sha256'):
        raise ValueError('The stored manifest content no longer matches its digest.')
    current = freeze_manifest(protocol, development_table)
    if current['sha256'] != manifest['sha256']:
        raise ValueError('Protocol or selected settings changed after the freeze.')


def final_audit(df, protocol, states, development_table, manifest):
    """Construct ALL predictions before evaluating the closed-form test target.

    Full-portfolio predictions retain simulated reference values at reference
    rows, then use each query's premium for dollar totals. The held-out test
    split contains no reference contracts. Simulated validation labels are
    not added to the reference training set after tuning.
    """
    verify_manifest(manifest, protocol, development_table)
    predicted = []
    for state in states:
        start = perf_counter()
        z_full = state['scaler'].transform(df[FEATURES])
        transform_s = perf_counter()-start
        rep_pos = df.index.get_indexer(state['representatives'].index)
        test_pos = df.index.get_indexer(state['parts']['test'].index)
        for family, fitted in state['models'].items():
            start = perf_counter()
            yhat = np.asarray(fitted['predict'](z_full), dtype=float)
            yhat[rep_pos] = state['yref']
            predict_s = perf_counter()-start
            row = {'compressor': state['compressor'], 'm': state['m'],
                   'seed': state['seed'], 'family': family, 'setting': fitted['name'],
                   'simulation_work': state['work']['pipeline_work'],
                   'validation_work': state['work']['validation_work'],
                   'reference_paths': state['work']['reference_paths'],
                   'end_to_end_s': state['common_seconds']+fitted['tuning_seconds']+transform_s+predict_s,
                   'final_prediction_s': transform_s+predict_s,
                   'metric_success': fitted['metric_success']}
            neighbor_position = None
            if family in ('CSM1NN', 'Learned1NN'):
                _, ni = nearest_predict(state['Zref'], state['yref'], z_full[test_pos],
                                        weights=fitted['weights'], k=1)
                neighbor_position = rep_pos[ni[:, 0]]
            predicted.append((row, yhat, test_pos, neighbor_position))
    # Direct MC simulates all N contracts at equal total path work, without tuning.
    direct_rows = []
    for seed in protocol['seeds']:
        n_paths = int(protocol['budget'])//len(df)
        start = perf_counter()
        labels = gmab_mc(df, n_paths, seed=int(seed)+90000)
        direct_seconds = perf_counter()-start
        test_pos = df.index.get_indexer(split_portfolio(df, int(protocol['split_seed']))['test'].index)
        direct_rows.append(({'compressor': 'none', 'm': len(df), 'seed': int(seed),
                             'family': 'MC-all', 'setting': f'{n_paths} paths/contract',
                             'simulation_work': n_paths*len(df), 'validation_work': 0,
                             'reference_paths': n_paths, 'end_to_end_s': direct_seconds,
                             'final_prediction_s': 0.0, 'metric_success': True},
                            labels['value'].to_numpy(), test_pos, None))
    predicted.extend(direct_rows)
    # The first access to final test values occurs HERE, after prediction construction.
    start = perf_counter()
    truth = gmab_exact(df)
    analytic_seconds = perf_counter()-start
    p = df['premium'].to_numpy()
    rows, prediction_rows = [], []
    for row, yhat, test_pos, neighbors in predicted:
        test_scores = score_predictions(truth[test_pos], yhat[test_pos], p[test_pos])
        for key in ['rmse', 'mae', 'p95_abs', 'bias', 'negative_fraction']:
            row['test_'+key] = test_scores[key]
        row['test_subportfolio_relative_error'] = test_scores['portfolio_relative_error']
        row['full_portfolio_relative_error'] = score_predictions(truth, yhat, p)['portfolio_relative_error']
        row['oracle_neighbor_gap_mae'] = (float(np.mean(np.abs(truth[test_pos]-truth[neighbors])))
                                          if neighbors is not None else np.nan)
        rows.append(row)
        prediction_rows.append(pd.DataFrame({
            'contract_id': df.iloc[test_pos]['contract_id'].to_numpy(),
            'compressor': row['compressor'], 'm': row['m'], 'seed': row['seed'],
            'family': row['family'], 'truth': truth[test_pos], 'prediction': yhat[test_pos],
            'premium': p[test_pos]}))
    return pd.DataFrame(rows), pd.concat(prediction_rows, ignore_index=True), {
        'analytic_audit_s': analytic_seconds, 'truth_full': truth,
        'audit_used_for_selection': False, 'analytic_available_for_this_teaching_model': True}


## 2. Reproduce development and verify the frozen settings

The default below matches Week 7. When conducting the team's own study, enter the
already-agreed protocol **before** running the final evaluation. This notebook includes
the full development procedure so it can reproduce the frozen settings without loading
a saved estimator or executing another notebook.

The final protocol includes all candidate families and comparisons. Validation selects
hyperparameters within each family. The test is used to evaluate these frozen families,
not to tune them or select additional feature engineering. A newly conceived post-test
experiment needs fresh held-out data and a separate exploratory label.

For the actual team study, paste Week 7's digest into `EXPECTED_WEEK7_SHA256` below.
An empty digest runs the standalone teaching example; it does not verify continuity
with a previous Week 7 run. With a recorded digest, disagreement stops the notebook
before final test values are opened. Identical software and settings are required.

In [ ]:
PROTOCOL = json.loads(json.dumps(DEFAULT_PROTOCOL))
EXPECTED_WEEK7_SHA256 = ''  # Paste the agreed Week 7 digest for the actual team study.
RUN_RESEARCH_SCALE = False
if RUN_RESEARCH_SCALE:
    PROTOCOL.update(n_contracts=10000, representative_counts=[100, 200],
                    seeds=[2027, 2028, 2029, 2030, 2031], budget=2_000_000,
                    n_validation_labels=100, validation_paths=2048)

portfolio = make_portfolio(PROTOCOL['n_contracts'], PROTOCOL['portfolio_seed'])
states, development, tuning, shared_development_s = run_development(portfolio, PROTOCOL)
manifest = freeze_manifest(PROTOCOL, development)
verify_manifest(manifest, PROTOCOL, development)
print('Frozen protocol:', manifest['sha256'])
if EXPECTED_WEEK7_SHA256 and manifest['sha256'] != EXPECTED_WEEK7_SHA256:
    raise ValueError('This run differs from the recorded Week 7 freeze. Resolve before testing.')
print('Week 7 continuity verified:' if EXPECTED_WEEK7_SHA256 else 'Standalone teaching run:',
      bool(EXPECTED_WEEK7_SHA256))
print('Test used to select settings:', manifest['test_used_for_selection'])
display(development[['compressor', 'm', 'seed', 'family', 'setting']].head(14))

## 3. Define the metrics before calculating them

For test contract $i$, let $e_i=\widehat y_i-y_i$ denote error per premium. We report

$$\operatorname{RMSE}=\sqrt{\frac1n\sum_i e_i^2},\qquad
\operatorname{MAE}=\frac1n\sum_i|e_i|,$$

along with the 95th percentile of $|e_i|$, signed mean bias, and the fraction of negative
predictions. Multiplying per-premium errors by 100 expresses **percentage points of
premium**, not a percentage error relative to a possibly tiny liability.

Dollar-weighted aggregate relative error on a set $S$ is

$$\frac{\sum_{i\in S} P_i(\widehat y_i-y_i)}{\sum_{i\in S}P_i y_i}.$$

We calculate it separately for the held-out test subportfolio and for all contracts.
The full-portfolio result includes reference contracts whose values were simulated.
It is therefore not a held-out prediction score. Positive and negative errors can
cancel in either aggregate: pair it with contract-level RMSE/MAE.

We avoid mean absolute percentage error because near-zero guarantee values make it
unstable. If an aggregate reference value is zero, its relative error is undefined.

In [ ]:
# All full-portfolio model predictions and direct-MC estimates are constructed
# inside final_audit before the first final analytic label is evaluated.
results, heldout_predictions, audit = final_audit(
    portfolio, PROTOCOL, states, development, manifest)
display(results[['compressor', 'm', 'seed', 'family', 'test_rmse', 'test_mae',
                 'test_p95_abs', 'test_bias', 'test_subportfolio_relative_error',
                 'full_portfolio_relative_error', 'simulation_work', 'end_to_end_s']])
print(f'Analytic audit of all {len(portfolio):,} contracts: {audit["analytic_audit_s"]:.6f} seconds')
print('Analytic labels used for selection:', audit['audit_used_for_selection'])

## 4. Summarize repeated runs without overstating uncertainty

The same test contracts are used across seeds and models, making comparisons paired.
Report mean and standard deviation across seeds. Two classroom repetitions are a
debugging-sized study, not strong statistical evidence. The optional research setting
uses five seeds; even then, seed variability on one synthetic portfolio does not prove
that a result generalizes to real insurance portfolios or other products.

The row for direct Monte Carlo simulates all contracts at the same total budget ceiling.
The pipeline cost includes representative selection, simulation, validation, fitting,
hyperparameter selection and full-portfolio prediction. Audit calculations are separate.
No negative regression prediction is silently discarded or clipped in the core comparison.

In [ ]:
summary = (results.groupby(['compressor', 'm', 'family'], as_index=False)
           .agg(rmse_mean=('test_rmse', 'mean'), rmse_sd=('test_rmse', 'std'),
                mae_mean=('test_mae', 'mean'), p95_mean=('test_p95_abs', 'mean'),
                bias_mean=('test_bias', 'mean'),
                full_portfolio_error_mean=('full_portfolio_relative_error', 'mean'),
                seconds_mean=('end_to_end_s', 'mean'),
                work_mean=('simulation_work', 'mean')))
display(summary.sort_values(['compressor', 'm', 'family']))

OUT = Path('outputs/week08')
OUT.mkdir(parents=True, exist_ok=True)
fig, ax = plt.subplots(figsize=(10, 4.8))
for family, group in summary.groupby('family'):
    ax.scatter(group['seconds_mean'], 100*group['rmse_mean'], s=55, label=family)
ax.set(xlabel='Standalone end-to-end cost (seconds)',
       ylabel='Test RMSE (percentage points of premium)',
       title=f'Frozen comparison at simulation-work ceiling B = {PROTOCOL["budget"]:,}')
ax.legend(ncol=1, fontsize=8, loc='upper left', bbox_to_anchor=(1.02, 1), borderaxespad=0)
fig.text(0.5, 0.015,
         f'Each surrogate point: one compressor/count configuration; means across {len(PROTOCOL["seeds"])} seeds.',
         ha='center', fontsize=8)
fig.tight_layout(rect=[0, 0.065, 1, 1])
fig.savefig(OUT/'accuracy_vs_runtime.png', dpi=180)
plt.show()

In [ ]:
# These illustrative panels were specified in advance, rather than chosen
# by whichever setting happens to have the smallest test error.
primary_m = max(PROTOCOL['representative_counts'])
primary_seed = PROTOCOL['seeds'][0]
primary_compressor = 'kmeans'
panels = ['CSM1NN', 'Learned1NN', 'RandomForest']
fig, axes = plt.subplots(1, 3, figsize=(12, 3.7), sharex=True, sharey=True)
for ax, family in zip(axes, panels):
    panel = heldout_predictions.query(
        'm == @primary_m and seed == @primary_seed and compressor == @primary_compressor and family == @family')
    ax.scatter(100*panel['truth'], 100*panel['prediction'], s=12, alpha=0.6)
    limits = [0, 100*max(panel['truth'].max(), panel['prediction'].max())*1.04]
    ax.plot(limits, limits, color='black', lw=1)
    ax.set(title=family, xlabel='Analytic value (% of premium)')
axes[0].set_ylabel('Predicted value (% of premium)')
fig.tight_layout()
fig.savefig(OUT/'heldout_prediction_scatter.png', dpi=180)
plt.show()

## 5. Did the learned distance select better neighbors?

For CSM-style and learned 1-NN, inspect the absolute difference between a test contract's
analytic normalized liability and its chosen representative's analytic liability.
This **oracle neighbor gap** is a post-freeze diagnostic, not a training target. It
removes representative-label noise from this one aspect of the comparison.

Separately inspect final prediction error, which still contains simulated label noise.
A smaller neighbor gap need not reduce runtime or dominate all direct predictors.
The learned weights describe this fitted approximation under the chosen scaling and
label budget. They are not causal effects or universal economic importance measures.

In [ ]:
neighbor_table = results[results['family'].isin(['CSM1NN', 'Learned1NN'])][
    ['compressor', 'm', 'seed', 'family', 'oracle_neighbor_gap_mae', 'test_mae', 'metric_success']]
display(neighbor_table)

weight_records = []
for state in states:
    fitted = state['models']['Learned1NN']
    for feature, weight in zip(FEATURES, fitted['weights']):
        weight_records.append({'compressor': state['compressor'], 'm': state['m'],
                               'seed': state['seed'], 'feature': feature,
                               'weight': float(weight), 'metric_success': fitted['metric_success']})
weights = pd.DataFrame(weight_records)
primary_weights = weights.query('m == @primary_m and compressor == @primary_compressor')
weight_summary = primary_weights.groupby('feature')['weight'].agg(['mean', 'std']).reindex(FEATURES)
fig, ax = plt.subplots(figsize=(8, 4.2))
ax.bar(weight_summary.index, weight_summary['mean'],
       yerr=weight_summary['std'].fillna(0), capsize=4, color='#35758a')
ax.axhline(1.0, color='black', linestyle='--', linewidth=1, label='Euclidean weight')
ax.set(ylabel='Learned diagonal weight',
       title=f'Learned weights: {primary_compressor}, {primary_m} representatives')
ax.tick_params(axis='x', rotation=20)
ax.legend()
fig.text(0.5, 0.015, 'Bars: mean across seeds; error bars: between-seed standard deviation.',
         ha='center', fontsize=8)
fig.tight_layout(rect=[0, 0.07, 1, 1])
fig.savefig(OUT/'learned_distance_weights.png', dpi=180)
plt.show()

## 6. Independently audit the Monte Carlo reference

An MC reference is an estimate, not ground truth. For a fixed subset of test contracts,
generate an independent high-precision reference and compare it with the analytic
formula. Approximate 95% intervals are $\widehat y\pm1.96\widehat{\mathrm{SE}}$.

The interval describes sampling uncertainty under the model, not model risk or the
uncertainty of a learned predictor. Finite-sample normal approximations may be poor
when very few paths produce a positive shortfall. Zero observed payoff variance does
not establish zero population liability. The synthetic core portfolio is chosen to
avoid spending the whole project on extremely rare events.

The MC audit is created after the freeze and its cost is kept separate. It does not
change model selection. Compare apparent error against the noisy MC reference with
error against the analytic value to see why this distinction matters.

In [ ]:
test_contracts = split_portfolio(portfolio, PROTOCOL['split_seed'])['test']
audit_contracts = test_contracts.iloc[:min(40, len(test_contracts))]
start = perf_counter()
audit_mc = gmab_mc(audit_contracts, n_paths=PROTOCOL['test_audit_paths'], seed=990001)
audit_mc_seconds = perf_counter()-start
audit_true = gmab_exact(audit_contracts)
audit_table = pd.DataFrame({
    'contract_id': audit_contracts['contract_id'].to_numpy(),
    'analytic_value': audit_true,
    'mc_value': audit_mc['value'].to_numpy(),
    'mc_se': audit_mc['se'].to_numpy()})
audit_table['covered_by_normal_95_interval'] = (
    np.abs(audit_table['mc_value']-audit_table['analytic_value']) <= 1.96*audit_table['mc_se'])
display(audit_table.head(10))
print('Observed coverage on this small audit sample:',
      audit_table['covered_by_normal_95_interval'].mean())
print(f'Independent MC audit: {len(audit_contracts)*PROTOCOL["test_audit_paths"]:,} payoff evaluations; '
      f'{audit_mc_seconds:.3f} seconds (excluded from every method)')

primary_prediction = heldout_predictions.query(
    'm == @primary_m and seed == @primary_seed and compressor == @primary_compressor and family == "Learned1NN"')
audit_comparison = primary_prediction.merge(audit_table, on='contract_id', validate='one_to_one')
comparison = pd.DataFrame([
    {'reference': 'analytic', **score_predictions(audit_comparison['analytic_value'], audit_comparison['prediction'])},
    {'reference': 'independent MC', **score_predictions(audit_comparison['mc_value'], audit_comparison['prediction'])},
])
display(comparison)

## 7. Save a reproducible evidence package

Save numerical results and the manifest rather than screenshots alone. The experiment
records software versions, seeds and model settings. All outputs below are generated
from the actual run. No tables have been prefilled with expected winners or speedups.

The main saved table has one row per method and repetition. Keep it even when a
result is disappointing. A summary table alone can hide instability, numerical failure,
negative predictions or cancellation in portfolio totals.

In [ ]:
results.to_csv(OUT/'final_results_per_run.csv', index=False)
summary.to_csv(OUT/'final_results_summary.csv', index=False)
heldout_predictions.to_csv(OUT/'heldout_predictions.csv', index=False)
weights.to_csv(OUT/'learned_weights.csv', index=False)
audit_table.to_csv(OUT/'independent_mc_audit.csv', index=False)
development.to_csv(OUT/'development_per_run.csv', index=False)
tuning.to_csv(OUT/'tuning_record.csv', index=False)
portfolio.to_csv(OUT/'synthetic_portfolio.csv', index=False)
audit_accounting = {
    'analytic_reference_seconds': audit['analytic_audit_s'],
    'independent_mc_reference_seconds': audit_mc_seconds,
    'independent_mc_reference_work': len(audit_contracts)*PROTOCOL['test_audit_paths'],
    'shared_development_seconds': shared_development_s}
(OUT/'frozen_protocol.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')
(OUT/'audit_accounting.json').write_text(json.dumps(audit_accounting, indent=2), encoding='utf-8')
print('Research outputs written to', OUT.resolve())
display(pd.DataFrame({'file': [p.name for p in sorted(OUT.iterdir()) if p.is_file()]}))

## 8. Write the report from evidence

A concise 4–6-page team report can use this structure:

1. **Question and motivation:** why representative valuation depends on a useful similarity rule.
2. **Model and scope:** GMAB top-up, fixed fee, risk-neutral terminal distribution, normalized target and assumptions.
3. **Methods:** compressor, MC labels, predictor families and the explicit learned distance.
4. **Experiment:** split, budgets, tuning, seeds, final freeze and accounting rules.
5. **Results:** one main table, error–cost figure, and neighbor/weight analysis; quantify uncertainty and failures.
6. **Limitations and conclusion:** the model has a closed form, the portfolio is synthetic,
   and current findings do not establish an industrial speedup or validity for other guarantees.

Describe an observation before explaining it. For example: report the observed paired
change in MAE, then examine neighbor gaps and MC SEs as possible explanations. Avoid
“AI is better” as a conclusion when the actual evidence concerns one model, one
simulation budget and one synthetic contract distribution.

Present positive and negative findings with the same care. A lower portfolio error
with a higher contract RMSE may reflect cancellation, not improved individual pricing.

## Reading assignment

Revisit selected passages from the existing five readings; no new technical paper is required.

* **Feng, Tan and Zheng (2020):** CSM and individual versus aggregate error.
  [Paper](https://www.math.uwaterloo.ca/~mbfeng/papers/2020_VAPort.pdf).
* **Gan and Lin (2015):** the representative-simulation-surrogate motivation, while
  recognizing that their functional/nested setting is more complicated.
  [Paper](https://www2.math.uconn.edu/~gan/ggpaper/gan2015ns.pdf).
* **Bauer, Kling and Russ (2008):** contract features left out of our teaching model.
  [Journal record](https://doi.org/10.2143/AST.38.2.2033356).
* **Hardy (2003), Investment Guarantees:** distinguish the guarantee liability from the
  investment account and explain why option-pricing assumptions matter.
* **Quan and Feng (2026), Warm-Started Projected Robbins-Monro for Portfolio-Scale
  Variable Annuity Fee Setting:** contrast value prediction with fee-root initialization;
  use the mentor-provided working manuscript.

Answer in 150–250 words: which conclusion is supported by your experiment, which is
only a plausible explanation, and which would require a different study?

## Coding and research homework

### Exercise 1 — Paired method comparison

Within each compressor/count/seed condition, subtract the CSM1NN test MAE from the
Learned1NN test MAE. Report the mean and standard deviation of these paired differences.
Explain the sign. Do not turn the repeated use of the same test contracts into a claim
of independent portfolio replications.

In [ ]:
paired_mae = results[results['family'].isin(['CSM1NN', 'Learned1NN'])].pivot(
    index=['compressor', 'm', 'seed'], columns='family', values='test_mae')
paired_mae['learned_minus_euclidean'] = paired_mae['Learned1NN']-paired_mae['CSM1NN']
display(paired_mae)
# Add a clearly labeled mean/SD summary and your interpretation.

### Exercise 2 — Cancellation and physical plausibility

Using the exported held-out predictions, identify a condition where aggregate error
and contract error tell different stories. Count negative predictions and describe
their implication. Do not silently clip them and replace the frozen result; a clipping
experiment would be a separately labeled follow-up on new held-out data.

In [ ]:
error_diagnostics = heldout_predictions.copy()
error_diagnostics['signed_error'] = error_diagnostics['prediction']-error_diagnostics['truth']
error_diagnostics['dollar_error'] = error_diagnostics['premium']*error_diagnostics['signed_error']
display(error_diagnostics[['family', 'contract_id', 'signed_error', 'dollar_error']].head())
# Group by the full condition keys and compare signed versus absolute errors.

### Exercise 3 — Prepare the research handoff

Finish the 4–6-page report and a presentation outline. Each student writes a short
contribution statement and checks one other student's section. Attach the environment,
manifest, code and result files so another person can reproduce the figures.

## Final research checklist

- [ ] All model and extension choices preceded the final test audit.
- [ ] Test-contract and full-portfolio metrics have different labels.
- [ ] MC noise, approximation error and modeling assumptions are distinguished.
- [ ] Whole-pipeline costs include fitting, selection and validation.
- [ ] Optimizer failures, negative predictions and unsuccessful methods are reported.
- [ ] The report contains measured results rather than a predetermined ranking.
- [ ] Files, reading responses and contribution statements are complete.

## Weeks 9 and 10 — Present the work

**Week 9:** assemble a complete presentation, rehearse to approximately 20 minutes,
check every numerical claim against a saved table, and practice a five-minute Q&A.
**Week 10:** revise the narrative, complete a second timed rehearsal, and ensure each
student can explain the pricing model, experiment and limitations. Confirm the final
event's allotted length with the program when announced; research talks may be longer.

The three students should each present a substantive part and answer questions across
the project. These two weeks add no new technical notebook.